# Phase 2: Predicting negative customer reviews

**Our research question:** Can we predict whether a delivered order will receive a negative review (1–2 stars)?

**Our stakeholder:** We focus on the customer experience and seller quality teams, who could use predicted risk to prioritise follow-up after delivery and before a review is submitted.

**Our target:** We define `is_negative_review = 1` for a 1–2 star review and `0` for a 3–5 star review.

**Our selected features:** We use 21 order-level predictors, grouped by the information they represent:

| Feature group | Features | Why we include them |
|---|---|---|
| Delivery performance | `delivery_days`, `days_early`, `late_delivery_flag` | We investigate whether delivery duration and arrival relative to the promised date are associated with dissatisfaction. Negative `days_early` means late; zero means on time; positive means early. |
| Order value and payment | `order_item_value`, `order_freight_value`, `payment_total`, `payment_count`, `max_installments`, `freight_share` | We examine whether order cost, shipping cost and payment characteristics help explain review risk. |
| Order and seller complexity | `item_count`, `seller_count`, `n_seller_states` | We test whether orders involving more items or sellers have different review outcomes. |
| Product characteristics | `primary_product_category`, `product_weight_g_mean`, `product_volume_cm3_mean`, `product_photos_qty_mean` | We examine whether the product mix and physical product characteristics contribute predictive information. |
| Geography | `customer_state`, `seller_state`, `same_state` | We account for broad customer and seller location differences. |
| Purchase timing | `purchase_month`, `purchase_weekday` | We allow for seasonal and weekday variation in order experiences. |

These are **candidate predictors**, not proven causes of negative reviews. In our modelling, we evaluate whether they help predict the target rather than assuming every feature is important.

**Our approach:** We compare Logistic Regression, Decision Tree and Random Forest, beginning with simple baselines. We use chronological train/test splitting, cross-validation and hyperparameter tuning, then evaluate the models before choosing one.

**Our prediction point:** We predict immediately after delivery, before a review has been submitted. We therefore exclude orders that were already reviewed before delivery. We do not use review scores, text or timestamps as predictors, because they would reveal information about the outcome.

We organise the notebook as follows:

1. Prepare our data and define the target.
2. Check leakage and split the data chronologically.
3. Compare simple baselines using cross-validation.
4. Tune Logistic Regression and Random Forest.
5. Evaluate our models on held-out orders.
6. Interpret feature importance.
7. Inspect prediction errors.
8. Explain our final choice, stakeholder use and limitations.

## 1. We prepare our data and define the target

We build **one row per delivered order** with a known review. We define `is_negative_review = 1` for 1–2 stars and `0` for 3–5 stars.

We use our existing `build_delivered_order_classification_dataset()` function to assemble validated order-level features. Our inputs describe delivery performance, order value, product details and geography.

**Why we filter:** We remove orders whose `review_answer_timestamp` occurred before `order_delivered_customer_date`. Our proposed model predicts at delivery, so it cannot intervene before a review that already exists. We use these two timestamps only for this historical eligibility check, **not** as model features.

In the code below, we print the population before and after filtering so we can explain exactly which orders we model.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.base import clone
from sklearn.metrics import (
    average_precision_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve, confusion_matrix,
)
from sklearn.inspection import permutation_importance
from sklearn.model_selection import TimeSeriesSplit, cross_validate, GridSearchCV

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.classification_data import (
    TARGET_COLUMN, DELIVERED_MODEL_FEATURES, build_delivered_order_classification_dataset,
    split_delivered_order_features_target,
)
from src.classification_model import build_delivered_order_logistic_pipeline
from src.classification_tree_model import (
    build_delivered_order_decision_tree_pipeline,
    build_delivered_order_random_forest_pipeline,
)

# We fix the random seed so our results can be reproduced.
SEED = 42
orders = pd.read_csv(ROOT / "data/processed/order_level.csv")
items = pd.read_csv(ROOT / "data/processed/item_level.csv")
# We reuse the team's validated order-level feature construction.
df = build_delivered_order_classification_dataset(orders, items)
lookup = orders.set_index("order_id").loc[df["order_id"]]
df["purchase_time"] = pd.to_datetime(lookup["order_purchase_timestamp"], format="mixed").to_numpy()
reviewed_at = pd.to_datetime(lookup["review_answer_timestamp"], format="mixed").to_numpy()
delivered_at = pd.to_datetime(lookup["order_delivered_customer_date"], format="mixed").to_numpy()
# We remove customers who had already reviewed before our prediction point.
already_reviewed = reviewed_at < delivered_at
print(f"Delivered orders with review: {len(df):,}")
print(f"Excluded (review before delivery): {already_reviewed.sum():,}")
df = df.loc[~already_reviewed].sort_values("purchase_time", kind="stable").reset_index(drop=True)
print(f"Eligible orders: {len(df):,}; negative review rate: {df[TARGET_COLUMN].mean():.1%}")
display(df[TARGET_COLUMN].value_counts().rename(index={0:"3–5 stars",1:"1–2 stars"}).to_frame("orders"))

## 2. We check leakage and split chronologically

We make predictions **after delivery, before review submission**. That lets us use delivery outcomes such as `days_early` and `late_delivery_flag`, but not information from the review itself. We use `split_delivered_order_features_target()` to keep the feature contract explicit.

We sort orders by `purchase_time` and reserve the newest 20% for our final test. We train and tune on the earlier 80%. This is a chronological proxy, not a complete simulation of every production event time.

We keep imputation, scaling and categorical encoding inside the scikit-learn pipelines. This matters because preprocessing statistics must be learnt from each training fold, not from future data.

In [ ]:
# We select the approved predictors and keep the review label separate.
X, y = split_delivered_order_features_target(df)
assert X.columns.tolist() == DELIVERED_MODEL_FEATURES
assert not any("review" in col for col in X.columns)
# We reserve the newest 20% as our untouched test period.
cut = int(len(df) * 0.80)
X_train, X_test = X.iloc[:cut], X.iloc[cut:]
y_train, y_test = y.iloc[:cut], y.iloc[cut:]
display(pd.DataFrame([
    {"population": "Train", "orders": len(y_train), "negative_rate": y_train.mean(),
     "first_purchase": df["purchase_time"].iloc[0].date(),
     "last_purchase": df["purchase_time"].iloc[cut-1].date()},
    {"population": "Test", "orders": len(y_test), "negative_rate": y_test.mean(),
     "first_purchase": df["purchase_time"].iloc[cut].date(),
     "last_purchase": df["purchase_time"].iloc[-1].date()},
]).round(3))

## 3. We compare simple baselines using cross-validation

We begin with our simplest variants: plain Logistic Regression and a single Decision Tree. We also run a default Random Forest so we can see whether combining trees improves on a single tree.

We use `TimeSeriesSplit(n_splits=5)` **within the training period**. The validation folds always follow their corresponding training folds in time.

We select models primarily by **PR AUC**, calculated using `average_precision`, because low reviews are the minority class. We also report `precision`, `recall`, `f1` and `roc_auc`. At this stage, threshold-dependent metrics use scikit-learn's default 0.50 decision cutoff.

Our aim is to show whether each added level of complexity earns its place, not to maximise the number of algorithms.

In [ ]:
# We perform time-ordered validation only on earlier training orders.
cv = TimeSeriesSplit(n_splits=5)
metrics = {"precision":"precision","recall":"recall","f1":"f1","roc_auc":"roc_auc","pr_auc":"average_precision"}
# We start with simple representatives from the two model families.
baseline = {
    "Logistic (plain)": build_delivered_order_logistic_pipeline(random_state=SEED).set_params(classifier__C=np.inf),
    "Decision Tree (plain)": build_delivered_order_decision_tree_pipeline(random_state=SEED),
    "Random Forest (default)": build_delivered_order_random_forest_pipeline(random_state=SEED, n_estimators=200),
}
rows = []
for name, pipeline in baseline.items():
    result = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=metrics, n_jobs=1)
    rows.append({"model":name, **{metric:result[f"test_{metric}"].mean() for metric in metrics}})
baseline_cv = pd.DataFrame(rows).set_index("model")
display(baseline_cv.round(3))

## 4. We tune our two main models

We tune only the two model families we plan to compare: Logistic Regression and Random Forest.

For Logistic Regression we search over `classifier__C` (regularisation strength) and `classifier__class_weight`. For Random Forest we vary `classifier__max_depth`, `classifier__min_samples_leaf` and `classifier__class_weight`.

We use `GridSearchCV` with `refit="pr_auc"`, so the best settings are chosen from **training cross-validation PR AUC**. We do not use the held-out test set to choose parameters or select our model. We print the best settings and a concise comparison table.

In [ ]:
# We tune only on training folds, using PR AUC to select settings.
logistic_grid = GridSearchCV(
    build_delivered_order_logistic_pipeline(random_state=SEED),
    {"classifier__C":[0.01,0.1,1.0,10.0],
     "classifier__class_weight":[None,"balanced"]},
    cv=cv, scoring=metrics, refit="pr_auc", n_jobs=1,
)
forest_grid = GridSearchCV(
    build_delivered_order_random_forest_pipeline(random_state=SEED,n_estimators=200),
    {"classifier__max_depth":[12,None],
     "classifier__min_samples_leaf":[10,25],
     "classifier__class_weight":[None,"balanced_subsample"]},
    cv=cv, scoring=metrics, refit="pr_auc", n_jobs=1,
)
logistic_grid.fit(X_train,y_train)
forest_grid.fit(X_train,y_train)
tuned = {"Logistic (tuned)":logistic_grid, "Random Forest (tuned)":forest_grid}
print("Selected parameters (training CV only):")
for name, search in tuned.items():
    print(name, search.best_params_)
tuned_cv = pd.DataFrame([
    {"model":name, **{metric:search.cv_results_[f"mean_test_{metric}"][search.best_index_] for metric in metrics}}
    for name,search in tuned.items()
]).set_index("model")
cv_results = pd.concat([baseline_cv,tuned_cv])
display(cv_results.sort_values("pr_auc",ascending=False).round(3))
# We lock our model choice before examining the test results.
chosen_name = cv_results["pr_auc"].idxmax()
print("Preselected model using CV PR AUC:",chosen_name)

## 5. We evaluate on later orders

Now we evaluate each fitted model once against our untouched `X_test` and `y_test`.

We calculate `predict_proba()[:, 1]` to obtain estimated negative-review risk. From those scores we report **PR AUC and ROC AUC**. For completeness we also calculate **precision, recall and F1** at a 0.50 threshold.

We use the model name chosen earlier from `cv_results["pr_auc"]`. We do **not** choose a different winner simply because it has a slightly higher score on our test set.

We plot ROC and precision–recall curves for the two tuned finalists. The prevalence of negative reviews in `y_test` provides a useful reference for PR AUC interpretation.

If a model predicts no negative cases at 0.50, we explain it as a threshold-dependent result rather than concluding that its probability ranking has no value.

In [ ]:
# We train each baseline on earlier orders and keep the test set for evaluation.
fitted = {name:clone(model).fit(X_train,y_train) for name,model in baseline.items()}
fitted.update({name:search.best_estimator_ for name,search in tuned.items()})
probs, test_rows = {}, []
for name,model in fitted.items():
    p = model.predict_proba(X_test)[:,1]
    # We use 0.50 solely for the threshold-based classification metrics.
    pred = (p>=0.5).astype(int)
    probs[name] = p
    test_rows.append({
        "model":name, "precision":precision_score(y_test,pred,zero_division=0),
        "recall":recall_score(y_test,pred,zero_division=0),
        "f1":f1_score(y_test,pred,zero_division=0),
        "roc_auc":roc_auc_score(y_test,p), "pr_auc":average_precision_score(y_test,p)
    })
test_results = pd.DataFrame(test_rows).set_index("model")
display(test_results.round(3))
print(f"Test prevalence, the random ranking PR reference: {y_test.mean():.3f}")
print("Model selected before opening the test set:",chosen_name)
fig, axes = plt.subplots(1,2,figsize=(12,4))
for name in tuned:
    fpr,tpr,_ = roc_curve(y_test,probs[name])
    precision,recall,_ = precision_recall_curve(y_test,probs[name])
    axes[0].plot(fpr,tpr,label=name)
    axes[1].plot(recall,precision,label=name)
axes[0].plot([0,1],[0,1],linestyle=":",label="Random")
axes[1].axhline(y_test.mean(),linestyle=":",label="Prevalence")
axes[0].set(title="ROC",xlabel="False positive rate",ylabel="True positive rate")
axes[1].set(title="Precision–recall",xlabel="Recall",ylabel="Precision")
for ax in axes:
    ax.legend()
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 6. We interpret what the model relies on

We use `permutation_importance()` on the selected model and report the 10 inputs with the biggest average decrease in `average_precision` when shuffled.

We interpret these values as **how strongly the fitted model depends on an input for prediction**, not proof that an input causes customers to leave low reviews.

We keep this analysis descriptive. We do not revisit hyperparameters based on test-set feature importance.

In [ ]:
# We shuffle each input to measure its effect on held-out PR AUC.
importance = permutation_importance(
    fitted[chosen_name], X_test, y_test, scoring="average_precision",
    n_repeats=3, random_state=SEED, n_jobs=1
)
feature_importance = pd.Series(
    importance.importances_mean,index=X_test.columns,name="PR AUC decrease"
).sort_values(ascending=False)
display(feature_importance.head(10).round(4).to_frame())

## 7. We inspect our model's errors

We create a confusion matrix using `confusion_matrix()` with an **illustrative 0.50 threshold**. This lets us see both missed negative reviews and false alarms.

We also group errors by delivery characteristics to check whether our model misses particular kinds of orders.

The 0.50 cutoff is not automatically the ideal business cutoff. If the customer experience team later sets a daily contact capacity or intervention policy, we would need to define and validate that decision rule separately.

In [ ]:
chosen_prob = probs[chosen_name]
# We use the same illustrative threshold for our error analysis.
pred = (chosen_prob>=0.5).astype(int)
display(pd.DataFrame(
    confusion_matrix(y_test,pred,labels=[0,1]),
    index=["Actual 3–5 stars","Actual 1–2 stars"],
    columns=["Predicted 3–5 stars","Predicted 1–2 stars"],
))
errors = X_test.copy()
errors["actual"] = y_test.to_numpy()
errors["predicted"] = pred
errors["result"] = np.select(
    [(errors["actual"]==1)&(errors["predicted"]==0),
     (errors["actual"]==0)&(errors["predicted"]==1)],
    ["Missed negative","False alarm"],default="Correct",
)
display(errors.groupby("result").agg(
    orders=("actual","size"), late_share=("late_delivery_flag","mean"),
    median_delivery_days=("delivery_days","median"),
).round(3))

## 8. We conclude and explain the limitations

**Our selection rule:** We choose `chosen_name` from the highest *training CV PR AUC*. In our Phase 2 report, we will compare both tuned finalists' CV and held-out metrics and explain whether any improvement warrants the added complexity. Similar test scores do not prove that the models are identical.

**How our stakeholder could use the result:** We could help the customer experience team prioritise newly delivered orders that have not already been reviewed. Our predictions rank risk; they do not prove that follow-up would prevent a negative review.

**Our limitations:** We use historical Olist data; some causes of dissatisfaction are not recorded; we exclude reviews submitted before delivery; and our purchase-time split is not a complete reproduction of real-world scoring conditions.

**Our next step:** Once we agree on our chosen model, we can retrain it and prepare it for the Phase 3 deployment.
